# Day 3 · Notebook 2 · The Seasonal Workbook 🎃
**Sobey HS Co-Op · Business Analytics & AI · Friday, October 30**

Every seasonal business asks the same three questions: **when does the season start, when does it peak, and how long does it last?**
This notebook answers them with two real datasets and the eight tools from Notebook 1, now on a table with 2,096 rows.

- **Google Trends, Canada**: how much Canadians searched for eight things, every week for five years. Scores run 0 to 100, where 100 is that search's busiest week.
- **NRF Halloween spending**: what US shoppers spend on Halloween each year since 2005, in total and per person.

**How to use it:** this one is run-along. Run every cell with **Shift + Enter**, in order. ▶ **RUN** cells just run; ✏️ **TRY** cells have one line to change.
Section 6 is yours: set `LANE` to your team's lane and every chart rebuilds. Stuck? **Runtime → Run all**. Sections: 0 set-up · 1 load · 2 clean and summarize · 3 the chart gallery · 4 honest charts · 5 season maths · 6 your lane · 7 credits.

## 0 · Set-up

In [ ]:
# Step 0 · ▶ RUN: the toolbox. pandas = tables, matplotlib + seaborn = charts
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import calendar, io, os, hashlib
print("pandas", pd.__version__, "· matplotlib", matplotlib.__version__, "· seaborn", sns.__version__)

In [ ]:
# ▶ RUN ONLY: the house chart style (SMU maroon and red, greys) and one colour per lane
MAROON, RED, INK, GREY, GRID = "#612141", "#9D2235", "#222222", "#5F6368", "#E6E6E6"
LANES = ["halloween costume", "candy", "pumpkin spice", "christmas gifts", "valentines day", "sunscreen", "snow tires", "back to school"]
COLOURS = dict(zip(LANES, ["#D55E00", "#9D2235", "#B8790F", "#2E8B57", "#CC79A7", "#56B4E9", "#0072B2", "#7B5EA7"]))  # colour-blind safe set
MONTHS = list(calendar.month_abbr)[1:]        # Jan ... Dec

plt.rcParams.update({
    "figure.figsize": (8, 4.5), "figure.dpi": 110, "font.size": 11,
    "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlecolor": MAROON,
    "axes.labelcolor": INK, "axes.edgecolor": GREY, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": GREY, "ytick.color": GREY, "legend.frameon": False,
})

def finish(ax, title, source, filename):
    """Every chart gets the same three things: a title that states the finding, a source line, a saved PNG."""
    ax.set_title(title)
    ax.figure.text(0.01, -0.04, "Source: " + source, fontsize=8.5, color=GREY, ha="left", va="top")
    ax.figure.savefig(filename, dpi=150, bbox_inches="tight")
    print("Saved", filename)

SRC_TRENDS = "Google Trends, Canada, web search, weekly, Oct 2021 to Oct 2026 (relative interest, 0 to 100 per lane)"
SRC_NRF = "NRF / Prosper Insights & Analytics annual Halloween survey (US consumers, US dollars; 2026 is the September forecast)"
print("House style ready:", ", ".join(LANES))

## 1 · Load and inspect
Both tables are **built into this notebook**, so nothing needs uploading. (If you did upload `Seasonal Trends Canada.csv` with the 📁 panel, the loader uses your file instead and says so.)
The loader also prints a **fingerprint**, a short code computed from the data's exact contents. Same fingerprint as the answer sheet = same data = same charts.

In [ ]:
# ▶ RUN ONLY: the data, built in. Scroll past; nothing to change here.
DATA_VERSION = "day3-seasonal-v3 (read Oct 29, 2026)"
EXPECTED_FINGERPRINT = "af2cf27c52e5"

BUILTIN_TRENDS = """week,halloween costume,candy,pumpkin spice,christmas gifts,valentines day,sunscreen,snow tires,back to school
2021-10-24,100,72,36,26,1,9,40,6
2021-10-31,26,75,25,42,1,10,47,7
2021-11-07,3,54,19,57,1,12,46,8
2021-11-14,2,52,12,67,1,11,49,7
2021-11-21,1,56,10,80,1,11,42,6
2021-11-28,1,60,9,86,1,11,46,6
2021-12-05,1,64,8,96,1,9,38,7
2021-12-12,1,68,8,91,1,9,27,9
2021-12-19,1,76,11,78,1,8,29,11
2021-12-26,1,86,5,10,2,10,35,90
2022-01-02,1,46,5,5,5,13,31,62
2022-01-09,1,40,5,3,9,14,16,64
2022-01-16,1,39,4,2,13,14,21,22
2022-01-23,1,44,4,1,18,17,16,9
2022-01-30,1,41,4,1,26,16,18,9
2022-02-06,1,42,4,1,53,18,12,7
2022-02-13,1,41,4,1,87,20,10,7
2022-02-20,1,43,4,1,2,23,13,7
2022-02-27,1,42,3,1,1,23,12,7
2022-03-06,1,43,3,2,1,27,10,7
2022-03-13,1,45,3,1,1,28,11,6
2022-03-20,1,42,2,1,1,30,12,8
2022-03-27,1,44,3,1,1,33,10,8
2022-04-03,1,44,3,1,1,36,9,8
2022-04-10,1,48,3,1,1,37,11,7
2022-04-17,1,42,3,1,1,36,9,8
2022-04-24,1,43,2,1,1,46,5,7
2022-05-01,1,42,2,1,1,47,7,7
2022-05-08,1,46,2,0,1,88,6,7
2022-05-15,1,44,2,1,0,57,5,7
2022-05-22,1,39,3,1,1,56,4,8
2022-05-29,1,44,2,1,1,65,4,10
2022-06-05,1,43,2,1,1,50,5,10
2022-06-12,1,42,3,1,0,48,4,13
2022-06-19,1,42,1,1,0,57,4,17
2022-06-26,1,42,2,1,0,58,4,13
2022-07-03,1,41,2,1,0,54,6,13
2022-07-10,2,41,2,2,0,58,5,15
2022-07-17,2,49,3,2,0,52,6,19
2022-07-24,3,50,5,2,1,51,6,27
2022-07-31,4,55,8,2,1,49,6,39
2022-08-07,5,47,12,2,0,39,7,53
2022-08-14,8,45,25,2,0,39,8,66
2022-08-21,10,44,52,2,0,31,9,78
2022-08-28,13,43,71,3,1,28,14,95
2022-09-04,16,40,53,4,0,25,17,89
2022-09-11,21,43,54,5,0,21,19,25
2022-09-18,23,44,65,6,0,19,21,16
2022-09-25,29,45,70,7,1,18,26,11
2022-10-02,42,49,96,10,1,18,30,7
2022-10-09,53,57,73,13,1,17,34,7
2022-10-16,61,58,47,14,1,17,40,7
2022-10-23,96,66,42,19,1,16,39,6
2022-10-30,45,86,31,29,1,17,49,8
2022-11-06,4,53,20,45,1,18,57,16
2022-11-13,2,54,13,61,1,17,73,7
2022-11-20,1,55,11,72,1,18,41,8
2022-11-27,2,61,8,78,1,17,61,6
2022-12-04,1,63,7,88,1,16,29,5
2022-12-11,1,69,8,100,1,15,37,5
2022-12-18,1,71,10,87,1,16,60,6
2022-12-25,1,57,6,19,2,21,27,14
2023-01-01,1,47,5,5,4,22,16,25
2023-01-08,1,44,4,3,8,25,12,11
2023-01-15,1,44,4,2,12,24,13,7
2023-01-22,1,47,4,2,15,24,17,6
2023-01-29,1,46,4,1,24,25,16,6
2023-02-05,1,50,4,1,39,25,9,6
2023-02-12,1,49,3,1,95,28,10,6
2023-02-19,1,45,3,1,2,32,13,5
2023-02-26,1,42,3,1,1,31,17,6
2023-03-05,1,45,4,1,1,33,11,7
2023-03-12,1,42,3,1,1,37,12,6
2023-03-19,1,44,2,1,1,39,12,6
2023-03-26,1,46,4,1,1,41,10,7
2023-04-02,1,44,3,1,1,40,11,7
2023-04-09,1,42,2,2,1,70,12,7
2023-04-16,1,45,3,2,1,56,8,6
2023-04-23,1,43,3,1,1,54,8,7
2023-04-30,1,46,2,2,1,58,7,7
2023-05-07,1,45,2,1,1,73,7,7
2023-05-14,1,45,3,1,1,78,5,8
2023-05-21,1,48,3,1,1,67,5,9
2023-05-28,1,41,2,2,1,93,4,9
2023-06-04,1,40,3,1,1,61,3,13
2023-06-11,1,41,2,2,1,51,7,15
2023-06-18,1,40,2,2,1,56,4,14
2023-06-25,1,40,2,2,1,54,5,12
2023-07-02,2,39,3,2,0,61,4,11
2023-07-09,2,40,3,2,1,62,9,14
2023-07-16,2,38,3,2,0,52,6,20
2023-07-23,3,38,4,3,1,50,6,25
2023-07-30,4,41,8,2,0,51,9,40
2023-08-06,5,43,14,2,0,40,8,50
2023-08-13,7,39,24,2,1,40,9,60
2023-08-20,9,43,66,3,1,33,13,68
2023-08-27,11,39,68,2,0,31,15,89
2023-09-03,14,36,69,4,1,31,17,93
2023-09-10,18,39,75,4,0,24,20,24
2023-09-17,22,41,78,5,1,24,26,14
2023-09-24,26,42,76,6,1,20,31,11
2023-10-01,33,43,100,8,1,20,33,8
2023-10-08,42,50,74,12,1,20,36,7
2023-10-15,51,57,50,13,1,19,42,7
2023-10-22,73,56,42,18,1,19,62,6
2023-10-29,46,81,33,25,1,17,61,7
2023-11-05,5,51,20,39,1,20,47,6
2023-11-12,2,51,15,50,1,19,37,6
2023-11-19,1,54,11,62,1,22,32,6
2023-11-26,1,66,9,72,1,20,41,7
2023-12-03,1,74,7,83,1,20,30,7
2023-12-10,1,80,7,91,1,20,24,7
2023-12-17,1,82,9,90,1,22,21,8
2023-12-24,1,69,9,27,2,23,14,12
2023-12-31,1,41,4,4,5,26,14,20
2024-01-07,1,37,3,3,8,26,35,11
2024-01-14,1,40,3,2,11,25,36,7
2024-01-21,1,41,3,1,15,29,15,7
2024-01-28,1,41,2,1,22,28,10,7
2024-02-04,1,44,3,1,33,30,10,6
2024-02-11,1,49,3,1,93,30,11,7
2024-02-18,1,40,3,1,3,36,11,5
2024-02-25,1,39,2,1,1,35,9,7
2024-03-03,1,45,2,1,1,39,15,6
2024-03-10,1,38,3,1,1,41,9,6
2024-03-17,1,41,3,1,1,43,11,6
2024-03-24,1,41,4,1,1,43,10,7
2024-03-31,1,43,3,1,1,46,12,7
2024-04-07,1,37,2,1,1,54,8,7
2024-04-14,1,40,3,1,1,53,6,7
2024-04-21,1,38,3,1,0,59,6,6
2024-04-28,1,39,2,1,0,54,6,6
2024-05-05,1,36,1,1,0,62,5,8
2024-05-12,1,37,1,1,0,65,6,7
2024-05-19,1,38,2,1,0,87,5,8
2024-05-26,1,39,2,1,0,71,5,9
2024-06-02,1,40,0,1,0,68,4,13
2024-06-09,1,39,2,1,0,63,5,18
2024-06-16,1,37,1,1,0,71,4,19
2024-06-23,1,39,2,1,0,63,4,11
2024-06-30,1,37,2,1,0,83,5,12
2024-07-07,2,35,2,1,0,78,5,14
2024-07-14,2,35,3,2,0,68,3,19
2024-07-21,3,35,5,1,0,60,7,25
2024-07-28,3,34,7,2,0,54,8,37
2024-08-04,5,36,12,2,1,44,6,49
2024-08-11,6,40,21,2,1,42,11,63
2024-08-18,9,38,63,3,1,34,13,76
2024-08-25,10,39,47,2,0,33,13,94
2024-09-01,14,38,58,4,1,30,17,86
2024-09-08,18,37,58,4,0,26,21,23
2024-09-15,20,39,57,5,1,26,18,14
2024-09-22,24,40,64,6,1,23,26,10
2024-09-29,30,44,69,8,1,22,35,9
2024-10-06,40,48,81,9,1,19,34,5
2024-10-13,44,48,65,14,1,20,38,6
2024-10-20,62,53,42,17,1,20,37,6
2024-10-27,71,93,33,22,1,19,39,7
2024-11-03,6,55,20,34,1,20,37,5
2024-11-10,2,51,14,49,1,21,37,6
2024-11-17,2,57,12,61,1,18,51,6
2024-11-24,1,56,10,73,1,20,53,5
2024-12-01,1,63,8,88,1,18,60,5
2024-12-08,1,66,7,91,1,19,37,6
2024-12-15,1,75,7,95,1,15,27,6
2024-12-22,1,65,10,51,2,20,23,9
2024-12-29,1,45,5,7,4,24,20,14
2025-01-05,1,36,6,3,8,22,16,14
2025-01-12,1,46,4,2,11,24,14,7
2025-01-19,1,41,3,1,14,24,12,6
2025-01-26,1,44,3,1,21,27,21,6
2025-02-02,1,41,4,1,32,28,27,6
2025-02-09,1,44,3,1,94,27,20,6
2025-02-16,1,34,3,1,5,31,23,6
2025-02-23,1,36,3,1,2,31,10,6
2025-03-02,1,37,3,1,1,32,11,6
2025-03-09,1,41,4,1,1,43,12,7
2025-03-16,1,40,4,1,1,43,11,7
2025-03-23,1,37,4,1,0,42,13,7
2025-03-30,1,39,3,1,1,44,11,8
2025-04-06,1,37,2,1,0,47,12,5
2025-04-13,1,40,5,1,0,55,8,7
2025-04-20,1,36,3,1,1,59,8,7
2025-04-27,1,35,2,1,0,63,6,6
2025-05-04,1,34,2,1,1,68,5,6
2025-05-11,1,37,2,1,0,73,5,7
2025-05-18,1,38,2,1,0,58,6,7
2025-05-25,1,38,3,1,0,66,6,9
2025-06-01,1,37,2,1,0,74,5,9
2025-06-08,1,39,3,1,1,68,5,11
2025-06-15,1,41,4,2,0,72,6,15
2025-06-22,1,38,3,1,0,72,6,12
2025-06-29,1,36,3,1,1,80,7,11
2025-07-06,2,36,4,1,0,72,7,14
2025-07-13,2,36,4,1,0,67,6,20
2025-07-20,3,37,6,1,1,59,8,27
2025-07-27,4,35,7,1,0,58,9,36
2025-08-03,5,39,13,1,0,54,8,50
2025-08-10,6,37,17,2,0,48,8,64
2025-08-17,8,38,51,2,0,38,10,75
2025-08-24,11,37,71,2,1,37,14,97
2025-08-31,14,55,60,3,1,32,17,96
2025-09-07,18,46,56,3,0,24,21,27
2025-09-14,21,38,59,4,1,25,20,16
2025-09-21,24,40,60,5,0,21,27,10
2025-09-28,32,42,65,6,0,23,32,10
2025-10-05,39,55,86,9,1,23,33,9
2025-10-12,46,68,65,12,0,21,44,12
2025-10-19,55,60,39,15,0,20,41,26
2025-10-26,82,100,33,19,0,21,45,58
2025-11-02,9,63,24,33,1,26,60,9
2025-11-09,2,53,17,46,1,28,100,9
2025-11-16,1,55,13,62,1,27,45,8
2025-11-23,1,61,11,76,1,29,53,10
2025-11-30,1,67,9,91,1,29,52,10
2025-12-07,1,72,8,95,1,28,47,11
2025-12-14,1,72,8,96,1,27,34,11
2025-12-21,1,76,9,72,1,24,30,11
2025-12-28,1,47,6,9,3,30,33,19
2026-01-04,1,44,4,4,8,33,28,22
2026-01-11,1,42,5,3,11,37,24,13
2026-01-18,1,41,4,2,14,37,25,14
2026-01-25,1,46,5,1,21,38,23,13
2026-02-01,1,49,5,2,32,44,14,13
2026-02-08,2,50,4,9,100,43,15,18
2026-02-15,2,45,5,8,15,47,15,18
2026-02-22,1,42,5,6,2,42,17,14
2026-03-01,1,43,4,3,1,44,15,12
2026-03-08,1,40,3,1,1,51,16,12
2026-03-15,1,44,4,1,1,57,14,12
2026-03-22,1,42,2,1,1,50,13,12
2026-03-29,1,45,4,1,1,53,17,11
2026-04-05,1,41,3,1,1,57,13,11
2026-04-12,1,38,3,1,1,58,12,12
2026-04-19,1,37,3,1,0,75,12,10
2026-04-26,1,51,3,3,1,73,9,11
2026-05-03,1,36,3,6,1,66,8,12
2026-05-10,1,46,2,4,1,64,6,14
2026-05-17,1,37,3,1,1,100,8,12
2026-05-24,1,35,3,1,1,94,9,13
2026-05-31,1,40,2,2,0,100,8,15
2026-06-07,1,35,2,1,1,88,6,16
2026-06-14,1,39,2,3,1,78,5,26
2026-06-21,1,39,3,3,1,95,6,34
2026-06-28,1,37,3,2,1,89,7,24
2026-07-05,2,56,4,2,1,90,7,27
2026-07-12,2,41,3,2,1,79,8,34
2026-07-19,3,41,6,2,1,61,10,33
2026-07-26,3,40,8,2,0,64,9,40
2026-08-02,5,40,12,2,0,58,10,54
2026-08-09,6,36,15,2,1,51,12,69
2026-08-16,8,42,32,3,1,51,15,77
2026-08-23,15,40,64,3,0,37,16,89
2026-08-30,13,39,54,4,0,37,18,100
2026-09-06,16,40,53,4,1,36,21,74
2026-09-13,19,39,55,6,1,32,22,27
2026-09-20,23,42,64,7,1,27,31,19
2026-09-27,30,45,72,8,1,26,36,15
2026-10-04,41,51,82,11,1,26,39,14
2026-10-11,46,52,63,13,1,24,44,11
2026-10-18,54,57,45,14,1,26,41,11
2026-10-25,86,71,38,17,1,24,51,12
"""

BUILTIN_SPENDING = """year,total_spending_billion_usd,per_person_usd,percent_celebrating,costumes_billion_usd,costumes_per_person_usd,costumes_percent_buying,candy_billion_usd,candy_per_person_usd,candy_percent_buying,decorations_billion_usd,decorations_per_person_usd,decorations_percent_buying,cards_billion_usd,cards_per_person_usd,cards_percent_buying
2005,,48.48,53,,,,,,,,,,,,
2006,,59.06,64,,,,,,,,,,,,
2007,,64.82,59,,,,,,,,,,,,
2008,,66.54,65,,,,,,,,,,,,
2009,,56.31,62,,,,,,,,,,,,
2010,5.8,66.28,64,2.0,23.37,62.0,1.8,20.29,96.0,1.6,18.66,68.0,0.3,3.95,33.0
2011,6.9,72.31,69,2.5,26.52,65.0,2.0,21.05,96.0,1.9,19.79,69.0,0.5,4.96,39.0
2012,8.0,79.82,72,2.9,28.65,66.0,2.3,23.27,96.0,2.4,23.56,73.0,0.6,4.34,41.0
2013,7.0,75.03,66,2.6,27.85,65.0,2.1,22.37,96.0,2.0,20.99,70.0,0.4,3.82,36.0
2014,7.4,77.52,67,2.8,29.26,67.0,2.2,23.39,95.0,2.0,21.2,69.0,0.4,3.67,36.0
2015,6.9,74.34,64,2.5,27.33,64.0,2.1,23.1,94.0,1.9,20.34,67.0,0.3,3.57,34.0
2016,8.4,82.93,69,3.1,31.03,67.0,2.5,24.43,94.0,2.4,23.62,70.0,0.4,3.84,35.0
2017,9.1,86.13,72,3.3,31.73,69.0,2.7,25.27,95.0,2.7,25.24,72.0,0.4,3.88,37.0
2018,9.0,86.79,70,3.2,31.26,68.0,2.6,25.47,95.0,2.7,26.21,74.0,0.4,3.85,35.0
2019,8.8,86.27,68,3.2,31.05,67.0,2.6,25.37,95.0,2.6,26.03,72.0,0.4,3.82,34.0
2020,8.0,92.12,58,2.6,29.9,65.0,2.4,27.55,96.0,2.6,29.63,75.0,0.4,5.03,40.0
2021,10.1,102.74,65,3.3,33.59,68.0,3.0,30.4,96.0,3.2,32.1,78.0,0.7,6.64,45.0
2022,10.6,100.45,69,3.6,33.75,67.0,3.1,29.51,96.0,3.4,31.89,75.0,0.6,5.3,39.0
2023,12.2,108.24,73,4.1,36.84,69.0,3.6,31.93,96.0,3.9,34.76,77.0,0.5,4.71,35.0
2024,11.6,103.63,72,3.8,33.92,67.0,3.5,31.69,95.0,3.8,33.57,75.0,0.5,4.45,33.0
2025,13.1,114.45,73,4.3,37.62,71.0,3.9,34.24,96.0,4.2,36.85,78.0,0.7,5.74,38.0
2026,13.5,115.14,74,4.4,37.32,71.0,4.1,34.79,96.0,4.3,36.61,78.0,0.8,6.43,42.0
"""

fingerprint = hashlib.sha256((BUILTIN_TRENDS + BUILTIN_SPENDING).encode("utf-8")).hexdigest()[:12]
print("Data version:", DATA_VERSION)
print("Fingerprint:", fingerprint, "(matches the answer sheet)" if fingerprint == EXPECTED_FINGERPRINT else "(DIFFERENT from the answer sheet: the data text was edited)")

In [ ]:
# Step 1a · ▶ RUN: load the Trends table (long form: one row per week per lane) and the spending table
wide = pd.read_csv(io.StringIO(BUILTIN_TRENDS))                       # one column per lane
trends = wide.melt(id_vars="week", var_name="lane", value_name="interest")   # one row per week per lane
source = "built-in copy"
if os.path.exists("Seasonal Trends Canada.csv"):                       # your uploaded file wins, if there is one
    trends = pd.read_csv("Seasonal Trends Canada.csv")[["week", "lane", "interest"]]
    source = "your uploaded Seasonal Trends Canada.csv"
spending = pd.read_csv(io.StringIO(BUILTIN_SPENDING))

print("Trends:", trends.shape[0], "rows ×", trends.shape[1], "columns, from the", source)
print("Spending:", spending.shape[0], "rows ×", spending.shape[1], "columns")
trends.head()

In [ ]:
# Step 1b · ▶ RUN: what is in the table? Column names, types, gaps
trends.info()
print()
print("Lanes:", list(trends["lane"].unique()))
print("First week:", trends["week"].min(), "· last week:", trends["week"].max())

In [ ]:
# Step 1c · ▶ RUN: the spending table has real gaps: NRF only reports totals by category from 2010
spending[["year", "total_spending_billion_usd", "per_person_usd", "percent_celebrating", "costumes_billion_usd", "candy_billion_usd"]].tail(8)

In [ ]:
# Step 1d · ▶ RUN: describe() = count, mean, min, quartiles, max in one line. Which lanes are "always on"?
trends.groupby("lane")["interest"].describe().round(1)

**What the numbers mean.** A lane's score is *relative to its own busiest week*: 100 is the peak, 50 is half as busy as the peak. So you can compare the *shape* of lanes, but a 40 for candy is not "the same amount of searching" as a 40 for sunscreen. Candy's mean of 47 says candy searches never die down; halloween costume's mean of 9 says it is almost silent for about 40 weeks a year.

## 2 · Clean and summarize
Three moves you will use all day: turn text dates into real dates, add helper columns (year, month), then **group by** something and summarize.

In [ ]:
# Step 2a · ▶ RUN: text dates become real dates; add year, month and a month name
trends["week"] = pd.to_datetime(trends["week"])
trends["year"] = trends["week"].dt.year
trends["month"] = trends["week"].dt.month
trends["month_name"] = trends["month"].apply(lambda m: MONTHS[m - 1])
print(trends.dtypes)
trends.sample(5, random_state=1)

In [ ]:
# Step 2b · ▶ RUN: group by lane and month: the average interest in each calendar month (the "seasonal calendar")
calendar_table = trends.groupby(["lane", "month"])["interest"].mean().unstack("month").round(0)
calendar_table.columns = MONTHS
calendar_table.loc[LANES]

In [ ]:
# Step 2c · ▶ RUN: group by lane and year: each lane's busiest week per year
peaks = trends.loc[trends.groupby(["lane", "year"])["interest"].idxmax(), ["lane", "year", "week", "interest"]]
peaks[peaks["lane"] == "halloween costume"]

**Read the peaks.** Halloween costume searches peak in the week that contains Halloween, every single year, and the score that week is the year's 100 or close to it. 2021 and 2026 are partial years (the data starts Oct 24, 2021 and ends Oct 25, 2026), which is why their rows look different. Always check how much of the year you actually have.

## 3 · The chart gallery: one question per chart
Seven chart types, each answering one question about the season. Every chart has a title that states the finding, a source line, and a saved PNG you can drop into a slide.

### Chart 1 · Line: *when* does Halloween start on Google?

In [ ]:
# Chart 1 · ▶ RUN: a line chart shows change over time. Two lanes, five years.
fig, ax = plt.subplots(figsize=(10, 4.5))
for lane in ["halloween costume", "candy"]:
    one = trends[trends["lane"] == lane]
    ax.plot(one["week"], one["interest"], color=COLOURS[lane], linewidth=1.8, label=lane)
ax.set_ylabel("search interest (0 to 100)")
ax.set_ylim(0, 122)                      # headroom so the legend never sits on a peak
ax.legend(loc="upper left", ncol=2)
finish(ax, "Costume searches spike for about a month; candy never goes quiet", SRC_TRENDS, "chart1 line halloween.png")

**What we can infer:** costume interest is near zero for ten months, climbs through September and explodes in the last two weeks of October, the same shape every year. Candy sits between 35 and 60 all year with two bumps: Halloween and Christmas. **What we can't infer:** how many people searched, or what they bought. Trends is relative interest, not sales.

### Chart 2 · Bar: which lanes are always-on and which are one-week wonders?

In [ ]:
# Chart 2 · ▶ RUN: a bar chart compares categories. Average weekly interest per lane, highest first.
means = trends.groupby("lane")["interest"].mean().sort_values(ascending=False)
fig, ax = plt.subplots()
ax.barh(means.index, means.values, color=[COLOURS[l] for l in means.index])
ax.invert_yaxis()
for i, v in enumerate(means.values):
    ax.text(v + 0.8, i, f"{v:.0f}", va="center", color=INK)
ax.set_xlabel("average weekly interest (0 to 100)")
finish(ax, "Candy and sunscreen are always-on; valentines day is a one-week wonder", SRC_TRENDS, "chart2 bar lane means.png")

**What we can infer:** the average tells you how *spread out* a season is. Candy (47) and sunscreen (40) sell for months; valentines day (4) and halloween costume (9) are almost all spike. A store in a spike lane lives or dies on timing. **Limit:** averages hide *when*; that is what the heatmap is for.

### Chart 3 · Grouped bar: where does the Halloween dollar go?

In [ ]:
# Chart 3 · ▶ RUN: grouped bars compare categories across years. NRF category totals, last five surveys.
cats = {"costumes_billion_usd": "costumes", "candy_billion_usd": "candy", "decorations_billion_usd": "decorations", "cards_billion_usd": "cards"}
recent = spending[spending["year"] >= 2022].set_index("year")[list(cats)].rename(columns=cats)
fig, ax = plt.subplots()
recent.plot.bar(ax=ax, color=["#D55E00", "#9D2235", "#2E8B57", "#7B5EA7"], width=0.8, rot=0)
ax.set_ylabel("US$ billions")
ax.set_ylim(0, 5)
ax.legend(ncol=4, loc="upper left", bbox_to_anchor=(0, 1.0))
ax.set_xlabel("")
finish(ax, "Costumes, candy and decorations each pass $4B in 2026; cards are the small slice", SRC_NRF, "chart3 grouped bar spending.png")

**What we can infer:** three categories of roughly equal size, all growing, with a dip in 2024 (the survey's $11.6B year). **Limit:** US shoppers, US dollars, a survey forecast made each September. Canada will rhyme with this, but we do not have the Canadian number.

### Chart 4 · Histogram: how many weeks a year are actually busy?

In [ ]:
# Chart 4 · ▶ RUN: a histogram counts how often each value happens. 262 weeks of two lanes, in bins of 5.
fig, ax = plt.subplots()
for lane in ["halloween costume", "candy"]:
    one = trends[trends["lane"] == lane]
    ax.hist(one["interest"], bins=range(0, 105, 5), alpha=0.65, color=COLOURS[lane], label=lane)
ax.set_xlabel("weekly search interest (0 to 100)")
ax.set_ylabel("number of weeks")
ax.legend()
finish(ax, "Costume interest is under 10 in 209 of 262 weeks; candy lives between 35 and 60", SRC_TRENDS, "chart4 histogram weeks.png")

**What we can infer:** the shape of a season. A tall bar at the left plus a long tail is a spike business; a hump in the middle is a steady one. **Ask:** how many weeks does the costume lane score above 50? (Count the tail: 12 weeks in five years, about two or three a year.)

### Chart 5 · Box plot: which month, and how sure are we?

In [ ]:
# Chart 5 · ▶ RUN: a box plot shows the spread of values in each group. Pumpkin spice, one box per month, five years of weeks.
one = trends[trends["lane"] == "pumpkin spice"]
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(data=one, x="month_name", y="interest", order=MONTHS, color=COLOURS["pumpkin spice"], width=0.6, ax=ax,
            flierprops={"marker": "o", "markersize": 4, "markerfacecolor": GREY, "markeredgecolor": "none"})
ax.set_xlabel("")
ax.set_ylabel("weekly search interest")
finish(ax, "Pumpkin spice season is September and October, with August as the ramp", SRC_TRENDS, "chart5 box pumpkin months.png")

**How to read a box:** the box is the middle half of the weeks; the line inside is the median (the typical week); the whiskers reach the rest of the usual weeks; dots are unusual weeks. August's box is short and low but its whisker is tall: most August weeks are quiet, a couple are already busy. That whisker is the *stock-up week*. **Limit:** five Augusts is a small sample; one strange year moves the box.

### Chart 6 · Scatter: do costume searches and candy searches move together?

In [ ]:
# Chart 6 · ▶ RUN: a scatter plot puts two measurements against each other, one dot per week.
pair = wide[["week", "halloween costume", "candy"]].copy()
r = pair["halloween costume"].corr(pair["candy"])
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(pair["halloween costume"], pair["candy"], s=22, alpha=0.6, color=MAROON, edgecolor="none")
ax.set_xlabel("halloween costume interest (same week)")
ax.set_ylabel("candy interest")
finish(ax, f"Weeks with more costume searching have more candy searching (r = {r:.2f})", SRC_TRENDS, "chart6 scatter costume candy.png")
print(f"Correlation r = {r:.2f} across {len(pair)} weeks")

**What we can infer:** when costume interest is high, candy interest is high too; the dots lean up and to the right and r is about 0.4, a moderate link, not a tight one: most weeks are quiet for both, and the busy costume weeks are only some of the busy candy weeks (candy has Christmas too). **What we can't infer:** that one causes the other. Both are caused by the calendar. "Correlation is not causation" is the most useful sentence in this course.

### Chart 7 · Heatmap: the whole seasonal calendar on one page

In [ ]:
# Chart 7 · ▶ RUN: a heatmap colours a table. Rows = lanes, columns = months, colour = average interest.
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.heatmap(calendar_table.loc[LANES], annot=True, fmt=".0f", cmap="Oranges", cbar_kws={"label": "average weekly interest"},
            linewidths=0.5, linecolor="white", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="y", rotation=0)
finish(ax, "Eight lanes, eight seasons: every month belongs to somebody", SRC_TRENDS, "chart7 heatmap lane month.png")

**What we can infer:** read across a row for a lane's season, down a column for who is busy this month. October belongs to costumes, pumpkin spice and snow tires; December to gifts and candy; February to valentines; May to July to sunscreen; August to back to school. **Limit:** monthly averages blur the exact week; section 5 finds the week.

## 4 · Honest charts
The same numbers can tell two stories. The chart on the left starts its axis at 9; the one on the right starts at 0. Only one of them is honest about how big the change is.

In [ ]:
# Chart 8 · ▶ RUN: NRF total Halloween spending, 2021 to 2026, drawn twice
recent = spending[spending["year"] >= 2021]
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, bottom, title in [(left, 9, "Spending EXPLODES (axis starts at 9)"), (right, 0, "Spending grew about a third since 2021 (axis starts at 0)")]:
    ax.bar(recent["year"].astype(str), recent["total_spending_billion_usd"], color=MAROON if bottom == 0 else RED)
    ax.set_ylim(bottom, 14.5)
    ax.set_ylabel("US$ billions")
    ax.set_title(title, fontsize=11.5)
    for x, v in zip(recent["year"].astype(str), recent["total_spending_billion_usd"]):
        ax.text(x, v + 0.12, f"${v}B", ha="center", fontsize=9, color=INK)
fig.suptitle("Same numbers, two stories: always check where the axis starts", x=0.01, y=1.04, ha="left", color=MAROON, fontweight="bold", fontsize=14)
fig.text(0.01, -0.04, "Source: " + SRC_NRF, fontsize=8.5, color=GREY, ha="left", va="top")
fig.savefig("chart8 honest axis.png", dpi=150, bbox_inches="tight")
print("Saved chart8 honest axis.png")
growth = recent["total_spending_billion_usd"].iloc[-1] / recent["total_spending_billion_usd"].iloc[0] - 1
print(f"2021 to 2026: {growth:.0%} growth")

**Four rules for every chart you make today:** 1) bar charts start at zero; 2) the title says the finding, not the topic; 3) a source line, always; 4) say the limit out loud (relative interest, not sales; US dollars; a survey). A chart without a limit is an ad.

## 5 · Season maths: the three numbers every pitch needs
Notebook 1's function, while loop and for loop, now on real data. For one lane in one year:
- **peak week**: the week with the highest score;
- **stock-up week**: walking *backwards* from the peak, the first week that is still at least `THRESHOLD` (50%) of the peak. Shelves must be full by then;
- **season length**: the stretch of weeks around the peak that stay at or above the threshold.

In [ ]:
# Step 5a · ✏️ TRY: the threshold. 0.5 = "half as busy as the peak". A florist might want 0.25; a costume shop 0.6.
THRESHOLD = 0.5

def season(one_year):
    """Peak week, stock-up week, last busy week and season length for the rows of ONE lane in ONE year."""
    values = list(one_year["interest"])
    weeks = list(one_year["week"])
    peak_i = values.index(max(values))               # position of the busiest week
    cut = THRESHOLD * values[peak_i]                 # the bar a week must clear to count as "in season"
    start = peak_i
    while start > 0 and values[start - 1] >= cut:    # walk backwards while weeks are still busy
        start = start - 1
    end = peak_i
    while end < len(values) - 1 and values[end + 1] >= cut:   # walk forwards the same way
        end = end + 1
    return {"peak_week": weeks[peak_i].date(), "peak": values[peak_i], "stock_up_week": weeks[start].date(),
            "last_busy_week": weeks[end].date(), "season_weeks": end - start + 1}

def season_table(lane):
    """One row per year for a lane: the season numbers plus how many weeks of data that year has."""
    rows = []
    one = trends[trends["lane"] == lane].sort_values("week")
    for year, part in one.groupby("year"):
        row = {"lane": lane, "year": year, "weeks_of_data": len(part), "full_year": len(part) >= 52}
        row.update(season(part))
        rows.append(row)
    return pd.DataFrame(rows)

season_table("halloween costume")

**Read it:** in every full year the costume lane's stock-up week is the first or second week of October, the peak is the week of Halloween, and the season is three or four weeks long. That is a decision: shelves full by October 5, ads from the last week of September, markdowns November 1. 2021 and 2026 are flagged `full_year = False`: partial years, handle with care.

In [ ]:
# Step 5b · ▶ RUN: the same function for every lane at once: how long is each season, and is it the same every year?
summary = pd.concat([season_table(lane) for lane in LANES])
full = summary[summary["full_year"]]
full.groupby("lane")["season_weeks"].agg(["min", "max", "mean"]).round(1).loc[LANES]

**What we can infer:** valentines day is a one-week season at the 50% rule; sunscreen runs 15 to 23 weeks; costume is 3 to 4 every year. "Same every year?" is answered by min versus max: a lane whose min and max match is predictable, which is exactly what a store wants. **Limit:** the threshold is a choice you made. Change it in 5a and the table changes; say so in your pitch.

## 6 · Your lane
Set `LANE` to your team's lane and run every cell below in order. Each one makes a chart or a table you can use in your three slides. Lanes: `halloween costume` · `candy` · `pumpkin spice` · `christmas gifts` · `valentines day` · `sunscreen` · `snow tires` · `back to school`.

In [ ]:
# Step 6 · ✏️ TRY: your lane (copy the name exactly, in quotes)
LANE = "halloween costume"

assert LANE in LANES, f"{LANE!r} is not a lane. Choose one of: {LANES}"
mine = trends[trends["lane"] == LANE].sort_values("week")
print(LANE, "·", len(mine), "weeks ·", mine["week"].min().date(), "to", mine["week"].max().date())

In [ ]:
# Step 6a · ▶ RUN: your lane over five years, with the stock-up week of each full year marked
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(mine["week"], mine["interest"], color=COLOURS[LANE], linewidth=1.8)
table = season_table(LANE)
for _, row in table[table["full_year"]].iterrows():
    ax.axvline(pd.Timestamp(row["stock_up_week"]), color=GREY, linestyle="--", linewidth=1)
ax.set_ylabel("search interest (0 to 100)")
ax.set_ylim(0, 122)
ax.text(0.99, 0.95, "dashed = stock-up week", transform=ax.transAxes, ha="right", color=GREY, fontsize=9)
finish(ax, f"{LANE}: the season repeats every year", SRC_TRENDS, f"lane line - {LANE}.png")

In [ ]:
# Step 6b · ▶ RUN: your lane by month: which months, and how sure?
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(data=mine, x="month_name", y="interest", order=MONTHS, color=COLOURS[LANE], width=0.6, ax=ax,
            flierprops={"marker": "o", "markersize": 4, "markerfacecolor": GREY, "markeredgecolor": "none"})
ax.set_xlabel("")
ax.set_ylabel("weekly search interest")
finish(ax, f"{LANE}: the busy months", SRC_TRENDS, f"lane box - {LANE}.png")

In [ ]:
# Step 6c · ▶ RUN: your lane as a heatmap, month by year: is the season drifting earlier or later?
grid = mine.groupby(["year", "month"])["interest"].mean().unstack("month").round(0)
grid.columns = [MONTHS[m - 1] for m in grid.columns]
fig, ax = plt.subplots(figsize=(10, 3.8))
sns.heatmap(grid, annot=True, fmt=".0f", cmap="Oranges", cbar=False, linewidths=0.5, linecolor="white", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="y", rotation=0)
finish(ax, f"{LANE}: month by year (blank = no data yet)", SRC_TRENDS, f"lane heatmap - {LANE}.png")

In [ ]:
# Step 6d · ▶ RUN: your season table: peak week, stock-up week, last busy week, season length, per year
table = season_table(LANE)
table

In [ ]:
# Step 6e · ▶ RUN: Halloween spending, for the three Halloween lanes (the others get a note instead)
SPENDING_COLUMN = {"halloween costume": "costumes_billion_usd", "candy": "candy_billion_usd", "pumpkin spice": "total_spending_billion_usd"}
if LANE in SPENDING_COLUMN:
    col = SPENDING_COLUMN[LANE]
    have = spending.dropna(subset=[col])
    fig, ax = plt.subplots()
    ax.plot(have["year"], have[col], marker="o", color=COLOURS[LANE], linewidth=2)
    ax.set_ylim(0, have[col].max() * 1.15)
    ax.set_ylabel("US$ billions")
    ax.set_xticks(have["year"][::2])
    finish(ax, f"{LANE}: US Halloween spending ({col.replace('_billion_usd', '').replace('_', ' ')}) has roughly doubled since 2010", SRC_NRF, f"lane spending - {LANE}.png")
    print(have[["year", col, "per_person_usd", "percent_celebrating"]].tail(5).to_string(index=False))
else:
    print(f"No NRF spending series for '{LANE}'. Your pitch uses the Trends charts; say so as a limit.")

In [ ]:
# Step 6f · ▶ RUN: save two small tables (under 30 rows) for your slides and for the dashboard skill
table.to_csv(f"season table - {LANE}.csv", index=False)
month_table = mine.groupby("month_name")["interest"].agg(["mean", "min", "max"]).round(1).reindex(MONTHS).reset_index().rename(columns={"month_name": "month"})
month_table.to_csv(f"month table - {LANE}.csv", index=False)
print("Saved", f"season table - {LANE}.csv", "and", f"month table - {LANE}.csv", "· open the 📁 panel on the left to download them")
month_table

**Your three slides come from here.** Slide 1: one cell explained (for example 5a, in your words). Slide 2: two or three charts from 6a to 6e, at least one of them a box plot, heatmap, histogram or scatter, plus one honest limit. Slide 3: the decision: stock date, ad start, markdown date, and how you would check it with real sales.

## 7 · Credits and limits
- **Google Trends** (trends.google.com): Canada, web search, weekly, "past 5 years", read on October 29, 2026, one search term per lane. Scores are *relative search interest*, 0 to 100 per term, not counts and not sales; Canada-wide, not Nova Scotia; a data-collection change applied from January 1, 2022.
- **NRF Halloween Data Center** (nrf.com): the National Retail Federation's annual Halloween consumer survey, conducted by Prosper Insights & Analytics. US consumers, US dollars; each year's figure is the forecast published in September of that year; category totals from 2010, per-person and participation from 2005.
- Charts: matplotlib and seaborn. House style: Saint Mary's University colours.